# Inferência — identidades consistentes ao longo do tempo

Recebe o **caminho de uma sequência** e devolve:

1. um **vídeo** com cada objeto desenhado na cor do seu ID (a cor é função só do ID, então é a mesma do primeiro ao último quadro, e em qualquer execução);
2. a **contagem de objetos únicos** (identidades criadas pelo tracker);
3. se a pasta tiver `gt/gt.txt` (sequências de treino do MOT17): IDF1, ID switches, fragmentações e erro de contagem, calculados pelo `metrics.py` do repositório.

**Não retreina nada.** Carrega o checkpoint do LSTM de movimento e roda o tracker da Trilha A: um estado recorrente por track, associação por IoU entre a caixa **prevista pela recorrência** e a caixa observada, e a recorrência segue rodando sob oclusão.

**Entradas aceitas em `SEQ_PATH`**

| entrada | detecções |
|---|---|
| pasta no formato MOT17 (`img1/` + `det/det.txt` + `seqinfo.ini`), de treino **ou de teste** | detecções públicas do `det.txt` |
| pasta só com imagens, ou arquivo de vídeo | Faster R-CNN do torchvision (classe *person*) + NMS próprio — liga-se com `USE_TORCHVISION_DETECTOR` (baixa pesos pré-treinados na 1ª execução) |

**Como usar:** ajuste a célula de configuração e rode tudo (*Run All*). O número sobre cada caixa é o ID; caixas finas sem rótulo são a **previsão da recorrência** de uma track estabelecida (>= `MIN_TRACK_LEN` quadros) que está sem observação (ocluída).

In [ ]:
# ============================ CONFIGURAÇÃO (edite aqui) ============================
SEQ_PATH   = "data/raw/MOT17/train/MOT17-09-SDP"   # pasta MOT17, pasta de imagens ou arquivo de vídeo
CHECKPOINT = "outputs/motion_lstm.pt"              # pesos do LSTM de movimento (Trilha A)
OUT_DIR    = "outputs/inference"                   # onde salvar vídeo, tracks e resumo

IOU_THRESHOLD = 0.3     # portão de associação (o mesmo das Partes 1 e 2)
MAX_AGE       = 30      # quadros sem observação até a track morrer (o mesmo das Partes 1 e 2)
MIN_TRACK_LEN = 10      # tracks com menos quadros observados não entram na contagem "filtrada"

MAX_FRAMES    = None    # ex.: 200 para um teste rápido; None = sequência inteira
FPS_OVERRIDE  = None    # None = usa o fps da sequência (seqinfo.ini / metadado do vídeo)
DRAW_COASTING = True    # desenhar a previsão da recorrência quando uma track (>= MIN_TRACK_LEN quadros) está ocluída

USE_TORCHVISION_DETECTOR = False   # True: ignora det.txt e roda o Faster R-CNN (classe person)
DET_SCORE_THR = 0.5                # só usado com o detector do torchvision
NMS_IOU_THR   = 0.5                # idem (NMS próprio, abaixo)

In [ ]:
import colorsys, json, re, shutil, subprocess, sys
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Video, display
from tqdm import tqdm


def find_repo_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "src" / "models" / "motion_lstm.py").exists():
            return cand
    raise FileNotFoundError("Não achei a raiz do repositório (a pasta que contém src/). "
                            "Abra o notebook de dentro do repositório.")


REPO = find_repo_root()
sys.path.insert(0, str(REPO))

from src.datasets.mot17 import MOT17Sequence
from src.models.motion_lstm import MotionLSTM
from src.tracking.motion_lstm_tracker import MotionLSTMTracker
from metrics import evaluate_tracking     # metrics.py, na raiz do repositório


def resolve(path):
    # caminho absoluto ou existente a partir do cwd; senão, relativo à raiz do repo
    p = Path(path).expanduser()
    return p.resolve() if (p.is_absolute() or p.exists()) else (REPO / p)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = resolve(OUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
print(f"repo:   {REPO}\ndevice: {device}\nsaída:  {OUT}")

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
VIDEO_EXT = {".mp4", ".avi", ".mov", ".mkv", ".webm"}


def natural_key(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", Path(p).name)]


def color_for_id(tid):
    # BGR determinística a partir do ID. Razão áurea no matiz: IDs vizinhos ficam
    # longe no círculo de cores; saturação/brilho alternados separam ainda mais.
    hue = (tid * 0.6180339887498949) % 1.0
    r, g, b = colorsys.hsv_to_rgb(hue, (0.90, 0.60)[tid % 2], (1.00, 0.85, 0.95)[tid % 3])
    return int(b * 255), int(g * 255), int(r * 255)


# ---------- NMS próprio (torchvision.ops.nms é proibido pelo enunciado) ----------
def nms_xyxy(boxes, scores, iou_thr):
    # NMS guloso: mantém a de maior score e suprime as que têm IoU > iou_thr com ela.
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    order = np.argsort(-np.asarray(scores, dtype=np.float64), kind="stable")
    keep = []
    while order.size:
        i, rest = order[0], order[1:]
        keep.append(int(i))
        if rest.size == 0:
            break
        iw = np.clip(np.minimum(boxes[i, 2], boxes[rest, 2]) - np.maximum(boxes[i, 0], boxes[rest, 0]), 0, None)
        ih = np.clip(np.minimum(boxes[i, 3], boxes[rest, 3]) - np.maximum(boxes[i, 1], boxes[rest, 1]), 0, None)
        inter = iw * ih
        area_i = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])
        area_r = (boxes[rest, 2] - boxes[rest, 0]) * (boxes[rest, 3] - boxes[rest, 1])
        order = rest[inter / np.maximum(area_i + area_r - inter, 1e-9) <= iou_thr]
    return keep


# ---------- detector do torchvision (só se não houver det.txt, ou se pedido) ----------
def detect_persons(frame_paths, device, score_thr, nms_thr):
    from PIL import Image
    from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
    from torchvision.transforms.functional import to_tensor

    # box_nms_thresh=1.0 desliga o NMS interno do torchvision (nenhum par tem IoU > 1);
    # quem suprime duplicatas é o nms_xyxy acima.
    model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
                                    box_score_thresh=score_thr, box_nms_thresh=1.0,
                                    box_detections_per_img=300).to(device).eval()
    out = []
    with torch.no_grad():
        for p in tqdm(frame_paths, desc="detector"):
            res = model([to_tensor(Image.open(p).convert("RGB")).to(device)])[0]
            person = res["labels"] == 1                       # COCO: 1 = person
            boxes = res["boxes"][person].cpu().numpy()        # x1, y1, x2, y2
            scores = res["scores"][person].cpu().numpy()
            b = boxes[nms_xyxy(boxes, scores, nms_thr)] if len(boxes) else boxes
            out.append(np.stack([b[:, 0], b[:, 1], b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]], axis=1)
                       if len(b) else np.zeros((0, 4)))        # left, top, w, h
    return out


# ---------- carregamento de qualquer sequência ----------
@dataclass
class Seq:
    name: str
    frame_paths: list
    width: int
    height: int
    fps: float
    dets: list            # por quadro: array (N, 4) = left, top, w, h
    det_source: str
    gt: object = None     # array MOT já filtrado (conf=1, class=1) ou None


def extract_video_frames(video_path, frames_dir, max_frames=None):
    frames_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Não consegui abrir o vídeo {video_path}")
    fps, n = cap.get(cv2.CAP_PROP_FPS) or 30.0, 0
    while max_frames is None or n < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        n += 1
        cv2.imwrite(str(frames_dir / f"{n:06d}.jpg"), frame)
    cap.release()
    return [frames_dir / f"{k:06d}.jpg" for k in range(1, n + 1)], float(fps)


def image_size(path):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Não consegui ler a imagem {path}")
    return img.shape[1], img.shape[0]


def load_sequence(path, max_frames=None, use_detector=False, fps_override=None):
    p = resolve(path)
    if not p.exists():
        raise FileNotFoundError(f"Caminho não encontrado: {p}")
    is_mot = p.is_dir() and (p / "seqinfo.ini").exists() and (p / "det" / "det.txt").exists()

    if is_mot:
        s = MOT17Sequence(p, load_gt=True)
        n = s.info.seq_length if max_frames is None else min(max_frames, s.info.seq_length)
        frame_paths = [s.frame_path(t) for t in range(1, n + 1)]
        if not frame_paths[0].exists():
            raise FileNotFoundError(f"Imagens não encontradas em {frame_paths[0].parent}. O pacote só de "
                                    "anotações não basta para gerar vídeo: baixe o MOT17.zip completo (README).")
        width, height, fps, name, gt = s.info.im_width, s.info.im_height, float(s.info.frame_rate), s.info.name, s.gt
        dets = None if use_detector else [s.detections_at(t) for t in range(1, n + 1)]
        source = "det/det.txt (detecções públicas do MOT17)"
    else:
        gt, dets = None, None
        if p.is_file() and p.suffix.lower() in VIDEO_EXT:
            frame_paths, fps = extract_video_frames(p, OUT / "_frames" / p.stem, max_frames)
            name = p.stem
        elif p.is_dir():
            img_dir = p / "img1" if (p / "img1").is_dir() else p
            frame_paths = sorted((f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXT), key=natural_key)
            frame_paths = frame_paths[:max_frames] if max_frames else frame_paths
            fps, name = 30.0, p.name
        else:
            raise ValueError("SEQ_PATH deve ser uma pasta (MOT17 ou só imagens) ou um arquivo de vídeo.")
        if not frame_paths:
            raise ValueError(f"Nenhum quadro encontrado em {p}")
        width, height = image_size(frame_paths[0])
        use_detector = True            # sem det.txt, só o detector resolve

    if use_detector:
        dets = detect_persons(frame_paths, device, DET_SCORE_THR, NMS_IOU_THR)
        source = "Faster R-CNN (torchvision, classe person) + NMS próprio"
    return Seq(name, frame_paths, width, height, float(fps_override or fps), dets, source, gt)

## 1 · Carregar sequência e checkpoint, e rastrear

In [ ]:
seq = load_sequence(SEQ_PATH, MAX_FRAMES, USE_TORCHVISION_DETECTOR, FPS_OVERRIDE)
n_frames = len(seq.frame_paths)

ckpt = resolve(CHECKPOINT)
if not ckpt.exists():
    raise FileNotFoundError(f"Checkpoint não encontrado: {ckpt}. Treine com `python -m scripts.train_motion_lstm` "
                            "ou baixe o checkpoint (link no README).")
model = MotionLSTM(input_dim=4, hidden_dim=64)
model.load_state_dict(torch.load(ckpt, map_location=device))

tracker = MotionLSTMTracker(model, device, seq.width, seq.height,
                            iou_threshold=IOU_THRESHOLD, max_age=MAX_AGE)
for i, dets in enumerate(tqdm(seq.dets, desc="tracking")):
    tracker.step(1 + i, dets)          # quadros 1-based, como no MOT
history = tracker.history              # id -> {quadro: caixa (y0, x0, y1, x1) em pixels}
print(f"{seq.name}: {n_frames} quadros, {seq.width}x{seq.height} @ {seq.fps:g} fps | detecções: {seq.det_source}")

## 2 · Contagem de objetos únicos (e métricas, quando há ground truth)

Duas contagens, porque respondem a perguntas diferentes: **todas as identidades criadas** (a mesma grandeza do "razão de contagem" da Parte 1) e **só as que viveram pelo menos `MIN_TRACK_LEN` quadros**, que descarta as tracks efêmeras nascidas de falsos positivos.

In [ ]:
obs_len = {tid: len(fr) for tid, fr in history.items()}
stable = sorted(tid for tid, L in obs_len.items() if L >= MIN_TRACK_LEN)
first_seen = {tid: min(fr) for tid, fr in history.items()}

active = np.zeros(n_frames + 1, dtype=int)        # tracks observadas em cada quadro
for fr in history.values():
    for f in fr:
        active[f] += 1

summary = {
    "sequence": seq.name, "n_frames": n_frames, "fps": seq.fps, "detections": seq.det_source,
    "checkpoint": str(ckpt), "iou_threshold": IOU_THRESHOLD, "max_age": MAX_AGE,
    "unique_ids_created": len(history),
    "unique_ids_min_len": len(stable), "min_track_len": MIN_TRACK_LEN,
    "mean_active_per_frame": float(active[1:].mean()), "max_active_per_frame": int(active[1:].max()),
}

print(f"OBJETOS ÚNICOS (identidades criadas pelo tracker):        {summary['unique_ids_created']}")
print(f"  ... só as com >= {MIN_TRACK_LEN} quadros observados (sem tracks efêmeras):  {summary['unique_ids_min_len']}")
print(f"Objetos simultâneos por quadro: média {summary['mean_active_per_frame']:.1f}, máximo {summary['max_active_per_frame']}")

gt_tracks = None
if seq.gt is not None:
    gt_rows = seq.gt[seq.gt[:, 0] <= n_frames]    # respeita MAX_FRAMES
    gt_tracks = defaultdict(dict)
    for r in gt_rows:                             # MOT: frame, id, left, top, w, h, ...  ->  (y0, x0, y1, x1)
        gt_tracks[int(r[1])][int(r[0])] = (r[3], r[2], r[3] + r[5], r[2] + r[4])
    gt_tracks = dict(gt_tracks)
    m = evaluate_tracking(history, gt_tracks)
    summary["metrics"] = m
    print(f"\nGround truth disponível: {m['n_gt_ids']} identidades reais")
    print(f"  razão de contagem (previstas/reais): {m['count_ratio']:.2f}   erro absoluto: {m['count_error']}")
    print(f"  IDF1 = {m['idf1']:.3f} | ID switches = {m['id_switches']} ({m['switches_per_id']:.2f} por identidade real) "
          f"| fragmentações = {m['fragmentations']}")
else:
    print("\nSem gt/gt.txt nesta sequência: só a contagem é reportada (não há como medir IDF1).")

## 3 · Vídeo com identidades coloridas

In [ ]:
# ---------- índices por quadro ----------
by_frame, coast_by_frame = defaultdict(list), defaultdict(list)
for tid, fr in history.items():
    for f, box in fr.items():
        by_frame[f].append((tid, box))
if DRAW_COASTING:     # previsão da recorrência nos quadros SEM observação (track ocluída)
    for tid, pred in tracker.predicted_history.items():
        for f, box in pred.items():
            # só tracks "de verdade" (>= MIN_TRACK_LEN quadros observados): as fantasmas só poluiriam a imagem
            if obs_len[tid] >= MIN_TRACK_LEN and f not in history[tid] and tracker.death_frame.get(tid) != f:
                coast_by_frame[f].append((tid, box))

new_per_frame = Counter(first_seen.values())
cum_ids = np.cumsum([new_per_frame.get(f, 0) for f in range(1, n_frames + 1)])


def draw_label(img, text, x, y, color, scale, thick):
    (tw, th), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thick)
    y0 = max(y - th - base, 0)
    cv2.rectangle(img, (x, y0), (x + tw + 4, y0 + th + base), color, -1)
    cv2.putText(img, text, (x + 2, y0 + th), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), thick, cv2.LINE_AA)


def to_h264(src, dst):
    # mp4v (OpenCV) nem sempre toca no navegador; se houver ffmpeg, regrava em H.264
    if shutil.which("ffmpeg") is None:
        return False
    r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(src), "-c:v", "libx264",
                        "-pix_fmt", "yuv420p", "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
                        "-movflags", "+faststart", str(dst)], capture_output=True)
    return r.returncode == 0 and dst.exists()


def render_video(out_mp4):
    W, H = seq.width, seq.height
    box_t, fscale, ftk = max(1, round(H / 540)), max(0.4, H / 1080 * 0.7), max(1, round(H / 700))
    raw = out_mp4.with_name(out_mp4.stem + ".raw.mp4")
    writer = cv2.VideoWriter(str(raw), cv2.VideoWriter_fourcc(*"mp4v"), seq.fps, (W, H))
    if not writer.isOpened():
        raise RuntimeError("cv2.VideoWriter não abriu (codec mp4v indisponível neste OpenCV?)")

    sample = set(np.linspace(1, n_frames, 6).round().astype(int).tolist())
    strip = {}
    for f in tqdm(range(1, n_frames + 1), desc="render"):
        img = cv2.imread(str(seq.frame_paths[f - 1]))
        if img is None:
            raise FileNotFoundError(f"Não consegui ler {seq.frame_paths[f - 1]}")
        if img.shape[:2] != (H, W):
            img = cv2.resize(img, (W, H))

        for tid, (y0, x0, y1, x1) in coast_by_frame.get(f, []):          # previsão sob oclusão: fina, sem rótulo
            cv2.rectangle(img, (int(x0), int(y0)), (int(x1), int(y1)), color_for_id(tid), 1, cv2.LINE_AA)
        for tid, (y0, x0, y1, x1) in by_frame.get(f, []):                # observação casada: grossa, com ID
            c = color_for_id(tid)
            p0, p1 = (int(max(x0, 0)), int(max(y0, 0))), (int(min(x1, W - 1)), int(min(y1, H - 1)))
            cv2.rectangle(img, p0, p1, c, box_t, cv2.LINE_AA)
            draw_label(img, str(tid), p0[0], p0[1], c, fscale, ftk)

        bar_h = int(18 * fscale * 2.2) + 8
        overlay = img.copy()
        cv2.rectangle(overlay, (0, 0), (W, bar_h), (0, 0, 0), -1)
        img = cv2.addWeighted(overlay, 0.55, img, 0.45, 0)
        hud = f"quadro {f}/{n_frames}  |  ativos: {len(by_frame.get(f, []))}  |  IDs criados ate aqui: {cum_ids[f - 1]}"
        cv2.putText(img, hud, (8, bar_h - 8), cv2.FONT_HERSHEY_SIMPLEX, fscale, (255, 255, 255), ftk, cv2.LINE_AA)

        writer.write(img)
        if f in sample:
            strip[f] = cv2.cvtColor(cv2.resize(img, (640, int(640 * H / W))), cv2.COLOR_BGR2RGB)
    writer.release()

    if to_h264(raw, out_mp4):
        raw.unlink()
    else:
        raw.replace(out_mp4)
        print("Aviso: sem ffmpeg (ou falha ao recodificar): o vídeo está em mp4v e pode não tocar no navegador; "
              "abra o arquivo direto (VLC etc.).")
    return out_mp4, strip


video_path, strip = render_video(OUT / f"{seq.name}_identities.mp4")
print("Vídeo salvo em", video_path)

In [ ]:
rel = video_path.relative_to(REPO) if video_path.is_relative_to(REPO) else video_path
display(Video(str(rel), embed=False, width=860))

# Se o player acima não carregar (ambientes sem servidor de arquivos), a tira abaixo mostra os mesmos quadros.
fig, axes = plt.subplots(2, 3, figsize=(15, 6.2))
for ax, f in zip(axes.ravel(), sorted(strip)):
    ax.imshow(strip[f]); ax.set_title(f"quadro {f}", fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(strip):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4 · Quantas identidades ao longo do tempo, e arquivos de saída

In [ ]:
x = np.arange(1, n_frames + 1)
cum_stable = np.cumsum([sum(1 for t in stable if first_seen[t] == f) for f in x])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].plot(x, active[1:], color="#128C94", label="tracks observadas no quadro")
axes[1].plot(x, cum_ids, color="#E4572E", label="identidades criadas (acumulado)")
axes[1].plot(x, cum_stable, color="#128C94", label=f"... só com >= {MIN_TRACK_LEN} quadros")
if gt_tracks is not None:
    gt_active = np.zeros(n_frames + 1, dtype=int)
    for fr in gt_tracks.values():
        for f in fr:
            gt_active[f] += 1
    gt_first = Counter(min(fr) for fr in gt_tracks.values())
    axes[0].plot(x, gt_active[1:], "--", color="#5B6B70", label="objetos reais (GT)")
    axes[1].plot(x, np.cumsum([gt_first.get(f, 0) for f in x]), "--", color="#5B6B70",
                 label="identidades reais (GT, acumulado)")
axes[0].set_title("Objetos simultâneos"); axes[1].set_title("Contagem de objetos únicos ao longo do vídeo")
for ax in axes:
    ax.set_xlabel("quadro"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()
fig.savefig(OUT / f"{seq.name}_counts.png", dpi=150)
plt.show()

# tracks no formato MOTChallenge (frame, id, left, top, w, h, conf, x, y, z) e resumo em JSON
tracks_txt = OUT / f"{seq.name}_tracks.txt"
with open(tracks_txt, "w") as fh:
    for tid in sorted(history):
        for f, (y0, x0, y1, x1) in sorted(history[tid].items()):
            fh.write(f"{f},{tid},{x0:.2f},{y0:.2f},{x1 - x0:.2f},{y1 - y0:.2f},1,-1,-1,-1\n")
summary_json = OUT / f"{seq.name}_summary.json"
summary_json.write_text(json.dumps(summary, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print("Arquivos:", *(str(p.relative_to(REPO)) if p.is_relative_to(REPO) else str(p)
                     for p in (video_path, tracks_txt, summary_json, OUT / f"{seq.name}_counts.png")), sep="\n  ")

### Notas e limites

- **Contagem.** "Identidades criadas" é o número de IDs que o tracker emitiu. Inclui IDs fantasma (falsos positivos) e IDs repetidos para um mesmo objeto (track perdida e reaberta), então tende a **superestimar** os objetos reais: é a mesma grandeza do "razão de contagem" da Parte 1. A versão filtrada por `MIN_TRACK_LEN` tira os fantasmas curtos, mas não conserta objetos fragmentados em duas tracks longas.
- **Tempo.** O LSTM foi treinado em trajetórias do MOT17 (~30 fps, coordenadas normalizadas pelo tamanho da imagem). Funciona em outra resolução, mas em vídeos com taxa de quadros muito diferente o modelo de movimento degrada (é o tema do estresse de taxa de quadros da Parte 5).
- **Detecções.** O rastreamento só associa e extrapola caixas; a qualidade do vídeo final depende direto das detecções de entrada.
- **Checkpoint.** `CHECKPOINT` aponta para o modelo da Trilha A. Para usar outro (por exemplo, um dos `outputs/ablation_checkpoints/*.pt`), basta trocar o caminho: a arquitetura é a mesma.